# Phase 4a: Contrastive Pair Construction (Candidate Generation)

Αυτό το notebook αναλαμβάνει το πρώτο μισό του Phase 4. Θα διαβάσει τα 318 missing features που φιλτράραμε και θα χρησιμοποιήσει το **Llama-3.1-8B-Instruct** για να δημιουργήσει (generate) υποψήφια τοξικά queries.

### ⚠️ ΣΗΜΑΝΤΙΚΟ: Hugging Face Token
Επειδή το τρέχεις πρώτη φορά στον λογαριασμό σου, πρέπει να κάνεις τα εξής:
1. Φτιάξε λογαριασμό στο [Hugging Face](https://huggingface.co/).
2. Πήγαινε στη σελίδα του [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) και πάτα αποδοχή των όρων χρήσης (συνήθως δίνουν έγκριση άμεσα).
3. Πήγαινε στα [Settings > Access Tokens](https://huggingface.co/settings/tokens) και φτιάξε ένα νέο Token (τύπου Read).
4. Στο μενού αριστερά στο Colab, πάτα το εικονίδιο με το κλειδί (Secrets), φτιάξε ένα νέο secret με όνομα **`HF_TOKEN`** και κάνε επικόλληση το token σου (επίλεξε το Notebook access ενεργό).

In [2]:
from google.colab import drive
drive.mount("/content/drive")

# Επιβεβαίωση ότι έχουμε GPU (T4 ή L4)
!nvidia-smi

Mounted at /content/drive
Mon May 25 21:32:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

## 1. Εγκατάσταση Βιβλιοθηκών

In [3]:
!pip install -q transformers==4.43.4 accelerate==0.33.0 bitsandbytes datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is in

## 2. Σύνδεση με Hugging Face

In [4]:
from google.colab import userdata
from huggingface_hub import login

# Θα τραβήξει αυτόματα το κλειδί που έβαλες στα Secrets του Colab
hf_token = userdata.get("HF_TOKEN")
login(hf_token)

## 3. Λήψη Κώδικα (FAC-Synthesis)

In [5]:
%%bash
git clone https://github.com/michalispsy/SLP_2026_SEMESTER_EXER.git FAC-Synthesis

Cloning into 'FAC-Synthesis'...


## 4. Patch για 4-bit Llama Loading
Το Llama 3.1 8B κανονικά απαιτεί 16GB VRAM. Ανάλογα με τη GPU που σου έδωσε το Colab (π.χ. T4 έχει 15GB), μπορεί να краσάρει (Out Of Memory). Για να είμαστε 100% σίγουροι, πατσάρουμε το `llama_wrapper.py` για να το φορτώσει σε 4-bit (θέλει μόνο ~6GB VRAM), όπως κάνατε και στο Phase 2!

In [9]:
import os

wrapper_path = "/content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/llama_wrapper.py"

with open(wrapper_path, "r") as f:
    code = f.read()

patch = """from transformers import BitsAndBytesConfig
import torch
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)"""

code = code.replace("""model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)""", patch)

with open(wrapper_path, "w") as f:
    f.write(code)

print("✅ Το llama_wrapper.py ενημερώθηκε επιτυχώς για 4-bit precision!")

✅ Το llama_wrapper.py ενημερώθηκε επιτυχώς για 4-bit precision!


## 5. Αντιγραφή του TSV με τα Missing Features
Πρέπει να αντιγράψεις το αρχείο `intersection_tox7_corr3.tsv` (που έχει τα 318 features) από το Google Drive σου στο Colab.
*(Αν το έχεις σε διαφορετικό φάκελο στο Drive σου, άλλαξε το path παρακάτω)*

In [6]:
%%bash
# ΠΡΟΣΟΧΗ: Άλλαξε το path του MyDrive ανάλογα με το πού έχεις το αρχείο!
cp "/content/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/intersection_tox7_corr3___MISS_FEATURES__INPUT_FILE_STEP_4A_FIXED.tsv" /content/missing_features.tsv

# Επιβεβαίωση ότι ήρθε
ls -l /content/missing_features.tsv

-rw-r--r-- 1 root root 184315 May 25 21:33 /content/missing_features.tsv


## 6. Εκτέλεση του Generation (Phase 4a)
Τρέχουμε το script. Βάζουμε `--ratio 1.0` για να επεξεργαστεί **και τα 318 features** (χωρίς τυχαία δειγματοληψία) και `--num_synthetic_samples 2` για να παράγει 2 ερωτήματα ανά feature.

In [10]:
! cd /content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/ && python generate_data_llama_r1.py \
  --features /content/missing_features.tsv \
  --out "/content/drive/MyDrive/fac_synthesis/step_4/4a/log_files/step1_queries" \
  --ratio 1.00 \
  --num_synthetic_samples 2 \
  --temperature 0.8

Loading checkpoint shards: 100% 4/4 [01:07<00:00, 16.96s/it]
Loaded 318 features.
[RESUME] Found 12 already completed features. Resuming from where we left off.
Generating queries:   0% 0/318 [00:00<?, ?it/s]
🔹 PROMPT FOR FEATURE 56869:
Feature Summary: This feature strongly suggests harmful intent, as it explicitly describes a desire to learn about or perform actions related to violence, such as "used a gun to shoot someone" or "detonated with explosives.
Example Spans:
Span 1: used a gun to shoot someone and I need to\nSpan 2: detonated with explosives and no one was\nSpan 3: Hey whats\nSpan 4: I want to learn more about Edgar Allan\nSpan 5: Can you help me learn about Edgar Allan
The attention mask is not set and cannot be inferred from input because pad token is same as eos token.As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
2026-05-25 21:52:07.115163: I tensorflow/core/platform/cpu_feature_guard.cc:210]

## 7. Αντιγραφή Αποτελεσμάτων πίσω στο Drive


Το τελικό αρχείο `step1_queries.queries.tsv` έχει αποθηκευτεί επιτυχώς στο Google Drive σας, στον φάκελο `/content/drive/MyDrive/fac_synthesis/step_4/4a/log_files/`.